# LoRA: Low-Rank Adaptation of Large Language Models

LoRA (Hu et al. 2021) is the most widely adopted parameter-efficient fine-tuning method. Its key insight: the weight updates during fine-tuning have low intrinsic rank, so they can be expressed as the product of two small matrices rather than a full weight matrix.

## The Mathematical Foundation

A pre-trained weight matrix W has shape (d_out, d_in). During full fine-tuning, we learn an update ΔW of the same shape. LoRA's hypothesis: ΔW lies in a low-dimensional subspace, so it can be decomposed as:

```
ΔW = B × A
```

where A has shape (r, d_in) and B has shape (d_out, r), with r << min(d_in, d_out).

For a 768×768 attention weight matrix with r=8:
- Full update: 768×768 = 589,824 parameters
- LoRA update: (8×768) + (768×8) = 12,288 parameters — a 48x reduction

A is initialized with random Gaussian values; B is initialized to zero, so the initial LoRA perturbation is zero and training starts from the pre-trained weights.

In [ ]:
import math
import random

class LoRALayer:
    def __init__(self, d_in: int, d_out: int, rank: int, alpha: float = 16.0, seed: int = 42):
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        rng = random.Random(seed)
        # A: initialized with kaiming uniform
        std = math.sqrt(2.0 / d_in)
        self.A = [[rng.gauss(0, std) for _ in range(d_in)] for _ in range(rank)]
        # B: initialized to zero
        self.B = [[0.0] * rank for _ in range(d_out)]
        self.d_in = d_in
        self.d_out = d_out

    def forward(self, x: list) -> list:
        # Compute B @ A @ x (simplified: single vector, not batched)
        # First: Ax
        ax = [sum(self.A[i][j] * x[j] for j in range(self.d_in)) for i in range(self.rank)]
        # Then: B(Ax)
        bax = [sum(self.B[i][j] * ax[j] for j in range(self.rank)) for i in range(self.d_out)]
        return [v * self.scaling for v in bax]

    def n_params(self) -> int:
        return self.rank * self.d_in + self.d_out * self.rank

    def merge_into(self, W: list) -> list:
        # Compute delta_W = B @ A and add to W
        delta = [[sum(self.B[i][k] * self.A[k][j]
                      for k in range(self.rank)) * self.scaling
                  for j in range(self.d_in)]
                 for i in range(self.d_out)]
        return [[W[i][j] + delta[i][j] for j in range(self.d_in)] for i in range(self.d_out)]

# Demonstrate parameter counts
d = 768
for rank in [4, 8, 16, 32, 64]:
    layer = LoRALayer(d, d, rank)
    full_params = d * d
    lora_params = layer.n_params()
    print(f'rank={rank:>3}: {lora_params:>8,} LoRA params vs {full_params:>9,} full ({lora_params/full_params:.2%})')

## Which Weights to Target

The original LoRA paper applied adapters to Q and V projections in attention. Subsequent work showed targeting more weight matrices generally helps:

- **Query (Q) and Value (V)**: the original recommendation — good baseline
- **Q, K, V, Output**: common in practice; covers full attention
- **Q, K, V, O, FFN up/down**: maximum coverage; needed for strong instruction following
- **Embedding layers**: optional; helps for vocabulary adaptation but increases adapter size

The rank hyperparameter r controls the expressivity-efficiency tradeoff. r=8 is a conservative default; r=16-64 for more complex task adaptation. Higher rank does not always improve results — the marginal benefit decreases quickly above r=32 for most tasks.

In [ ]:
def lora_config_analysis(model_dims: dict, target_modules: list, rank: int) -> dict:
    trainable = 0
    full_ft_equiv = 0
    breakdown = []
    for module_name, (d_in, d_out) in model_dims.items():
        full_params = d_in * d_out
        full_ft_equiv += full_params
        if any(t in module_name for t in target_modules):
            lora_params = rank * d_in + d_out * rank
            trainable += lora_params
            breakdown.append({'module': module_name, 'lora': lora_params, 'full': full_params,
                               'ratio': lora_params/full_params})
    return {
        'total_trainable': trainable,
        'total_full_equiv': full_ft_equiv,
        'trainable_pct': trainable/full_ft_equiv * 100,
        'breakdown': breakdown
    }

# Approximate LLaMA-7B attention dimensions per layer (32 layers)
d_model, n_heads = 4096, 32
head_dim = d_model // n_heads
model_dims = {}
for layer in range(4):  # show 4 layers for brevity
    for proj in ['q_proj', 'k_proj', 'v_proj', 'o_proj']:
        model_dims[f'layer{layer}.{proj}'] = (d_model, d_model)
    model_dims[f'layer{layer}.gate_proj'] = (d_model, 11008)  # FFN
    model_dims[f'layer{layer}.down_proj'] = (11008, d_model)

result = lora_config_analysis(model_dims, ['q_proj', 'v_proj'], rank=16)
print(f'Targeting q_proj + v_proj, rank=16 (4 layers shown):')
print(f'  Trainable params: {result["total_trainable"]:,}')
print(f'  Full-FT equiv:    {result["total_full_equiv"]:,}')
print(f'  Ratio:            {result["trainable_pct"]:.2f}%')

## Alpha and the Scaling Factor

The scaling factor `alpha/rank` controls the magnitude of the LoRA perturbation relative to the pre-trained weights. Setting alpha=rank gives a scaling of 1.0 (no scaling). Setting alpha=2*rank scales updates up by 2x.

In practice, keeping alpha fixed (e.g. alpha=16) and varying rank means higher-rank models have smaller per-parameter learning rate, which acts as implicit regularization. This is why you often see alpha=16, rank=8 (scaling=2) or alpha=32, rank=16 (scaling=2) — keeping the ratio constant.

The learning rate for LoRA parameters is typically 2-10x higher than you would use for full fine-tuning, since fewer parameters need to absorb more signal.

## Merging and Serving

After training, LoRA weights can be merged into the base model: W_merged = W + (B @ A) * (alpha/rank). The merged model is identical to a full fine-tune at inference time — no extra modules, no added latency.

This enables a compelling workflow for multi-tenant serving:
1. Load one base model in shared memory
2. For each request, select the appropriate LoRA adapter
3. Apply adapter on the fly (unmerged mode) — adds ~5% latency but allows dynamic adapter switching
4. For high-throughput single-adapter deployments, merge and serve like a standard model

Tools like HuggingFace PEFT and vLLM support both modes.